In [1]:
import pandas as pd

for split in ["train", "validation", "test"]:
    df = pd.read_parquet(f"{split}.parquet", columns=["user_id", "product_id", "event_type"])
    print(f"\n=== {split} ===")
    print("Rows:", len(df))
    print("Unique users:", df["user_id"].nunique())
    print("Unique products:", df["product_id"].nunique())
    print(df["event_type"].value_counts())


=== train ===
Rows: 29218702
Unique users: 2312200
Unique products: 153182
event_type
view        28059856
cart          642499
purchase      516347
Name: count, dtype: int64

=== validation ===
Rows: 6897095
Unique users: 839354
Unique products: 121229
event_type
view        6595724
cart         180674
purchase     120697
Name: count, dtype: int64

=== test ===
Rows: 6332967
Unique users: 764234
Unique products: 120920
event_type
view        6123819
purchase     105805
cart         103343
Name: count, dtype: int64


In [ ]:

import pandas as pd


def extract_features(events: pd.DataFrame) -> pd.DataFrame:
    events = events.copy()
    events["event_time"] = pd.to_datetime(events["event_time"])
    events["user_id"] = events["user_id"].astype(str)
    events["product_id"] = events["product_id"].astype(str)

    reference_time = events["event_time"].max()
    print(f"Reference time (latest event in file): {reference_time}")

    views = events[events["event_type"] == "view"]
    carts = events[events["event_type"] == "cart"]

    # user_id + product_id level features 
    last_view = (
        views.groupby(["user_id", "product_id"])["event_time"].max()
        .rename("last_view_time")
    )
    last_cart = (
        carts.groupby(["user_id", "product_id"])["event_time"].max()
        .rename("last_cart_time")
    )
    session_count = (
        events.groupby(["user_id", "product_id"])["user_session"]
        .nunique()
        .rename("session_count")
    )

    #  user-level features
    user_total_views = views.groupby("user_id").size().rename("user_total_views")
    user_unique_products = (
        views.groupby("user_id")["product_id"].nunique().rename("user_unique_products")
    )

    # product-level features 
    product_total_carts = carts.groupby("product_id").size().rename("product_total_carts")

    # most recent known price per product
    price_lookup = (
        events.sort_values("event_time")
        .groupby("product_id")["price"]
        .last()
        .rename("price")
    )

    #  build one row per user-product pair that had any interaction 
    result = (
        events[["user_id", "product_id"]]
        .drop_duplicates()
        .set_index(["user_id", "product_id"])
    )

    result = result.join(last_view, how="left")
    result = result.join(last_cart, how="left")
    result = result.join(session_count, how="left")
    result = result.reset_index()

    result["days_since_last_view"] = (
        (reference_time - result["last_view_time"]).dt.total_seconds() / 86400
    )
    result["days_since_last_cart"] = (
        (reference_time - result["last_cart_time"]).dt.total_seconds() / 86400
    )

    result = result.merge(user_total_views, on="user_id", how="left")
    result = result.merge(user_unique_products, on="user_id", how="left")
    result = result.merge(product_total_carts, on="product_id", how="left")
    result = result.merge(price_lookup, on="product_id", how="left")

    result["session_count"] = result["session_count"].fillna(0).astype(int)
    result["user_total_views"] = result["user_total_views"].fillna(0).astype(int)
    result["user_unique_products"] = result["user_unique_products"].fillna(0).astype(int)
    result["product_total_carts"] = result["product_total_carts"].fillna(0).astype(int)

    result = result.drop(columns=["last_view_time", "last_cart_time"])

    return result


if __name__ == "__main__":
    
    RAW_FILE_PATH = "train.parquet"
    OUTPUT_PATH = "train_features.parquet"

    events = pd.read_parquet(RAW_FILE_PATH)
    features_df = extract_features(events)

    print("\nSample of extracted features:")
    print(features_df.head())
    print(f"\nTotal user-product pairs: {len(features_df):,}")
    print(f"\nMissing values per column:")
    print(features_df.isna().sum())

    features_df.to_parquet(OUTPUT_PATH, index=False)
    print(f"\nSaved to {OUTPUT_PATH}")

Reference time (latest event in file): 2019-10-21 23:59:59+00:00

Sample of extracted features:
     user_id product_id  session_count  days_since_last_view  \
0  541312140   44600062              1             20.999815   
1  554748717    3900821              1             20.999988   
2  519107250   17200506              1             20.999977   
3  550050854    1307067              1             20.998414   
4  535871217    1004237              1             20.999942   

   days_since_last_cart  user_total_views  user_unique_products  \
0                   NaN               122                    87   
1                   NaN                 3                     2   
2                   NaN                21                    15   
3                   NaN                19                    15   
4                   NaN                68                    21   

   product_total_carts        price  
0                    0    35.790001  
1                   22    33.200001  
2 